# T18 / E09 — Tinh chỉnh ba bộ mã hóa làm mốc so sánh

Đây là **mốc so sánh nghiêm túc nhất** của đề tài. Khác baseline tầm thường ở E01, ba mô hình
này thật sự đọc văn bản; khác LLM giám khảo ở E10, chúng chạy cục bộ không tốn API. Nếu phương
pháp chú ý nội tại không vượt được chúng thì lập luận về chi phí ở câu hỏi CH2 không đứng vững.

**Notebook settings:**

- Accelerator: **GPU T4 x2**
- Internet: **On**
- Data: attach dataset `unicorn1209/vihallulens`
- Add-ons → Secrets: `HF_TOKEN` (không bắt buộc)

## Ước tính thời gian

| Mô hình | Độ dài | Batch | 1 seed | 3 seed |
|---|---|---|---|---|
| PhoBERT-large | 256 | 16 | ~7 phút | ~25 phút |
| XLM-R-large | 512 | 8 | ~25 phút | ~80 phút |
| InfoXLM-large | 512 | 8 | ~25 phút | ~80 phút |

**Tổng khoảng 3 giờ.** Hạn mức 30 giờ/tuần nên thoải mái, nhưng đừng chạy cả ba trong một phiên
— xem ô 4.

## Chạy một lần cả ba, hay tách ba phiên?

**Chạy một lần cả ba là được**, và đó là mặc định của notebook này.

Hạ xung vì nhiệt (đo ở T08, mục 5 `CLAUDE.md`) làm card **chậm đi** 10–15 % khi chạy liên tục,
nhưng nó **không làm sai kết quả**. Toàn bộ macro-F1, accuracy, F1 từng lớp — thứ mà mốc so
sánh này sinh ra để đo — không hề bị ảnh hưởng.

Thứ duy nhất bị ảnh hưởng là cột **`ms/mẫu`**: mô hình chạy sau trông chậm hơn thực lực. Mà cột
đó dùng cho E11 để so **nhóm phương pháp** (bộ mã hóa vs chú ý nội tại vs Gemini), không phải
so PhoBERT với XLM-R. Chênh lệch giữa ba mô hình vốn đã tới từ độ dài chuỗi — 256 với 512
token, tức khoảng gấp đôi — nên 15 % hạ xung không đổi thứ hạng hay bậc độ lớn.

**Script tự đo nhiệt độ và xung nhịp**, in cảnh báo nếu phát hiện hạ xung, và ghi số liệu vào
`results/runs.jsonl`. Nên chạy xong sẽ biết chắc có bị hay không thay vì phải phỏng đoán.

Tách ba phiên riêng chỉ đáng làm nếu sau này cần so `ms/mẫu` giữa ba mô hình với nhau một cách
chặt chẽ. Lúc đó chạy lại từng cái là được, điểm số không phải chạy lại.

In [ ]:
# Ô 1 — lấy code. Chạy lại được nhiều lần.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
    print("đã cập nhật repo có sẵn")
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))
    print("đã clone mới")

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

In [ ]:
# Ô 2 — cài đặt. pyvi là phụ thuộc mới của T18, dùng để tách từ cho PhoBERT.
!pip install -q --no-deps -e .
!pip install -q -U transformers accelerate pyvi

In [ ]:
# Ô 3 — chuẩn bị dữ liệu. E09 đọc data/interim nên phải chuẩn hóa và chia tập trước.
# Chạy trên CPU, khoảng 2 phút. Copy output dán vào PR.
import os

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: đã nạp từ Kaggle Secrets")
except Exception:
    print("HF_TOKEN: không có, ba mô hình đều mở nên vẫn chạy được")

get_ipython().system("python scripts/probe_env.py")
get_ipython().system("python scripts/normalize_data.py --dataset vihallu")
get_ipython().system("python scripts/split_data.py")
get_ipython().system("python -m pytest tests/test_encoder.py -q")

## Ba mô hình

Chạy cả ba ô liền nhau cũng được — xem phần trên về hạ xung. Mỗi ô ghi kết quả vào
`results/runs.jsonl` ngay khi xong, nên nếu ô sau hỏng thì kết quả ô trước vẫn còn.

In [ ]:
# Ô 4 — PhoBERT-large. Khoảng 25 phút với 3 seed.
!python scripts/train_encoder_baseline.py --model phobert

In [ ]:
# Ô 5 — XLM-R-large. Khoảng 80 phút với 3 seed.
!python scripts/train_encoder_baseline.py --model xlmr

In [ ]:
# Ô 6 — InfoXLM-large. Khoảng 80 phút với 3 seed.
!python scripts/train_encoder_baseline.py --model infoxlm

In [ ]:
# Ô 7 — lấy kết quả về. results/ không đẩy ngược lên GitHub từ notebook được,
# nên tải file này xuống rồi commit từ máy cá nhân.
import shutil

shutil.copy("results/runs.jsonl", "/kaggle/working/runs.jsonl")
with open("results/runs.jsonl", encoding="utf-8") as handle:
    print(handle.read())

## Đọc kết quả thế nào

**Mốc phải vượt là 0,702**, không phải 0,670. Đó là cận trên khoảng tin cậy của baseline tầm
thường E01 — đo ở T17. Một mô hình đạt 0,68 chỉ là nằm trong khoảng nhiễu của hai đặc trưng bề
mặt, chưa nói lên điều gì.

Bốn con số cần chú ý:

1. **macro-F1 trung bình qua các seed**, kèm khoảng tin cậy 95 % từ bootstrap tập test.
2. **F1 của lớp `intrinsic`.** E01 chỉ đạt 0,523 ở lớp này và bắt đúng 45,2 % — đây là lớp khó
   nhất. Bộ mã hóa cải thiện được bao nhiêu ở đây mới là phần đáng nói.
3. **Tỷ lệ mẫu bị cắt vì quá dài.** Đo trước khi chạy: khoảng **50 % cặp của ViHallu vượt giới
   hạn 256 token của PhoBERT**, trong khi ở 512 token chỉ khoảng 1,4 %. Nếu PhoBERT thua thì
   phải ghi rõ nó thua một phần vì **không đọc hết được ngữ cảnh**, chứ không kết luận là mô
   hình yếu hơn.
4. **VRAM đỉnh và ms/mẫu**, cho bảng đánh đổi chi phí ở E11.

5. **Dòng cảnh báo hạ xung** ở cuối mỗi mô hình. Nếu có, điểm số vẫn dùng được bình thường,
   chỉ cột `ms/mẫu` là không so thẳng được với mô hình chạy ở phiên khác.

## Nếu hết bộ nhớ

Giảm batch trước, đừng giảm độ dài — độ dài đang là chỗ PhoBERT vốn đã thiệt. Sửa `MODELS`
trong `src/vihallulens/detect/encoder.py`.

Muốn theo đúng nguyên văn quy tắc "5 seed" ở mục 3 `docs/EXPERIMENTS.md` thì thêm `--seeds 5`,
đổi lại mỗi mô hình 512 token tốn thêm khoảng một giờ.